<a href="https://colab.research.google.com/github/NVIDIA-NeMo/DataDesigner/blob/main/docs/colab_notebooks/8-generating-word-documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Data Designer Tutorial: Generating Word Documents

#### 📚 What you'll learn

Your document pipeline probably does not ingest parquet. It ingests `.docx`.

This notebook builds a corpus of synthetic corporate policy documents and writes each row out as a real
Word file — the kind of dataset an enterprise RAG index, a document classifier, or a DLP scanner needs to
be developed against, and the kind you are never allowed to copy out of a customer's SharePoint.

- 🧱 **Structure, not prose**: use a Pydantic model as the LLM's `output_format` so you never parse markdown
- 💬 **Field descriptions are prompt text**: why `Field(description=...)` is the fastest quality lever you have
- 🎲 **Labelled by construction**: sampler controls become dataset columns, so no annotation pass is needed
- 🖨️ **Rendering**: turn each row into a `.docx` with headings, tables, footers, and Word core properties

> **Prerequisites**: This notebook uses [build.nvidia.com](https://build.nvidia.com/models) and
> [`python-docx`](https://python-docx.readthedocs.io/). The setup cells below install the dependencies
> and pick up your API key.

This notebook builds on the [structured outputs tutorial](https://docs.nvidia.com/nemo/datadesigner/tutorials/structured-outputs-jinja-expressions-and-conditional-generation).
If this is your first time using Data Designer, start with the
[first notebook](https://docs.nvidia.com/nemo/datadesigner/tutorials/the-basics) in this series.


### ⚡ Colab Setup

Run the cells below to install the dependencies and set up the API key. If you don't have an API key, you can generate one from [build.nvidia.com](https://build.nvidia.com).


In [ ]:
%%capture
!pip install -U data-designer "python-docx>=1.1.0,<2"

In [ ]:
import getpass
import os

from google.colab import userdata

try:
    os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
except userdata.SecretNotFoundError:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA API key: ")

### 📦 Import Data Designer

- `data_designer.config` provides the configuration API.
- `DataDesigner` is the main interface for generation.


In [ ]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

In [ ]:
data_designer = DataDesigner()

### 🎛️ Define model configurations

- Writing a whole document in one structured call is a long generation, so `max_tokens` matters more here
  than in most tutorials.

- If the model runs out of tokens mid-JSON you do not get a short document, you get a parse failure on the
  column. That is the good outcome: it fails loudly instead of silently truncating.


In [ ]:
MODEL_ALIAS = "doc-writer"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model="nvidia/nemotron-3-super-120b-a12b",
        provider="nvidia",
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=0.9,
            top_p=0.95,
            max_tokens=8192,
        ),
    )
]

## 🧱 Model the document, not the prose

The tempting approach is to ask for "a policy document", get back a wall of markdown, and then write a
parser that hunts for `##` and pipe tables and turns them into Word styles.

That parser is where the project dies. Models are inconsistent about markdown in exactly the ways that
break naive parsers, and every failure mode you fix creates a new regex.

So we never generate prose-with-structure in the first place. We generate the **structure**, with prose
inside it.


In [ ]:
from pydantic import BaseModel, Field


class DocTable(BaseModel):
    """A simple rectangular table."""

    caption: str = Field(description="Short caption describing what the table contains.")
    columns: list[str] = Field(description="Column headers, 2 to 4 of them.")
    rows: list[list[str]] = Field(description="Table rows. Each row has one cell per column header.")


class DocSection(BaseModel):
    """One numbered section of the document."""

    heading: str = Field(description="Section heading, title case, no numbering prefix.")
    paragraphs: list[str] = Field(
        description="One to three body paragraphs of prose. No markdown, no bullet characters."
    )
    bullets: list[str] = Field(
        default_factory=list,
        description=(
            "Optional bulleted requirements or steps for this section. Use an empty list when the "
            "section reads better as prose only."
        ),
    )


class WordDocument(BaseModel):
    """A complete business document, structured for rendering."""

    title: str = Field(description="Document title.")
    subtitle: str = Field(description="One-line subtitle, e.g. the scope or the owning function.")
    summary: str = Field(description="A single paragraph executive summary, 40-80 words.")
    sections: list[DocSection] = Field(description="Four to six sections that make up the body.")
    key_data: DocTable = Field(
        description=(
            "A table carrying the document's structured facts — thresholds, review cadences, "
            "roles and responsibilities, retention windows, or similar."
        )
    )

This one model does two jobs. It is the `output_format` of an `LLMStructuredColumnConfig`, and it is the
input contract of the renderer we write further down.

Because both ends share one definition, *"the LLM produced something the renderer can't handle"* stops
being a category of bug you defend against with heuristics. It becomes a Pydantic validation error on the
column, which Data Designer already knows how to retry.

> 💡 **Your field descriptions are prompt text**
>
> Data Designer serializes the JSON Schema into the prompt inside `<response_schema>` tags and asks for a
> fenced `json` block. Every `Field(description=...)` you write is shipped to the model verbatim.
>
> `description="Column headers, 2 to 4 of them."` is not documentation. It is the instruction that stops
> you getting nine-column tables. Write those descriptions like prompts, because that is what they are.


## 🎲 Design the corpus with samplers

Every document will be joined to a row recording the controls that produced it. *"Give me the Restricted
Finance documents"* becomes a dataframe filter instead of an annotation project — the labels are free
because they came first.

Note `doc_type` is a **subcategory** of `department`, so a Finance document is a "Revenue Recognition
Policy" and never an "Adverse Event Reporting Procedure".


In [ ]:
DOC_TYPES_BY_DEPARTMENT = {
    "Information Security": ["Access Control Standard", "Incident Response Runbook"],
    "Human Resources": ["Remote Work Policy", "Travel and Expense Policy"],
    "Finance": ["Revenue Recognition Policy", "Month-End Close Runbook"],
    "Procurement": ["Vendor Onboarding Procedure", "Contract Renewal Policy"],
}

config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="doc_id",
        sampler_type=dd.SamplerType.UUID,
        params=dd.UUIDSamplerParams(prefix="POL-", short_form=True, uppercase=True),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="company",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Northwind Diagnostics", "Cobalt Ridge Financial", "Halden Biopharma"]
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="department",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(values=list(DOC_TYPES_BY_DEPARTMENT)),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="doc_type",
        sampler_type=dd.SamplerType.SUBCATEGORY,
        params=dd.SubcategorySamplerParams(category="department", values=DOC_TYPES_BY_DEPARTMENT),
    )
)

# Skewed on purpose: a realistic corpus is mostly Internal with a thin tail of
# Restricted documents, and that tail is usually the interesting test slice.
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="classification",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Public", "Internal", "Confidential", "Restricted"],
            weights=[1, 6, 3, 1],
        ),
    )
)

## ✍️ Generate the document

One LLM call per document. Everything above was samplers, which are free.

The `{% if %}` block is what makes `classification` more than a label sitting next to the document — it
becomes a claim about the body that an extraction pipeline can be scored against.


In [ ]:
config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="document",
        model_alias=MODEL_ALIAS,
        output_format=WordDocument,
        prompt=(
            "Write an internal {{ doc_type }} for {{ company }}, owned by the {{ department }} "
            "department. The document ID is {{ doc_id }}.\n\n"
            "Write in the flat, procedural register of a real corporate policy document. No marketing "
            "language, no first person. Sections must be specific to a {{ doc_type }} — scope, roles, the "
            "actual procedure, exceptions, enforcement — not filler like 'Introduction'.\n"
            "{% if classification in ['Confidential', 'Restricted'] %}"
            "This document is {{ classification }}. State the handling restrictions explicitly in the "
            "scope section.\n"
            "{% endif %}"
            "The key_data table must carry concrete, checkable facts — thresholds, timeframes, "
            "role-to-responsibility mappings — not prose chopped into cells.\n"
            "Body text is plain prose. No markdown, no '**', no bullet characters inside paragraphs."
        ),
    )
)

data_designer.validate(config_builder)

### 🔁 Iteration is key – preview the dataset!

Check that the sections are specific to the document type, that the table holds facts rather than chopped
up prose, and that the classification shows up in the scope section when it should.


In [ ]:
preview = data_designer.preview(config_builder, num_records=2)

In [ ]:
preview.display_sample_record()

## 🖨️ Render to `.docx`

Now `python-docx` turns a `WordDocument` into a real Word file.

Note what this function does **not** do: it does not parse anything. It walks a validated Pydantic object,
because all the ambiguity was removed one step earlier by the schema.


In [ ]:
import re
from pathlib import Path

from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt

UNSAFE_FILENAME_CHARS = re.compile(r"[^A-Za-z0-9._-]+")


def safe_filename(name: str) -> str:
    """Turn an arbitrary string into something safe to write to disk."""
    stem = UNSAFE_FILENAME_CHARS.sub("-", name.strip()).strip("-._")
    return (stem[:120] or "document") + ".docx"


def normalize_rows(table: DocTable) -> list[list[str]]:
    """Pad or truncate every row to the header width.

    Structured outputs constrain the *shape* of the JSON, not the arithmetic inside it. A model told to
    produce a three-column table will occasionally hand back a row with two cells — nothing in the JSON
    Schema forbids it. Retrying costs a whole document generation; padding costs four lines.
    """
    width = len(table.columns)
    normalized = []
    for row in table.rows:
        cells = [str(cell) for cell in row][:width]
        cells.extend([""] * (width - len(cells)))
        normalized.append(cells)
    return normalized


def add_table(doc: Document, headers: list[str], rows: list[list[str]]) -> None:
    """Append a table with a bold header row."""
    table = doc.add_table(rows=1, cols=max(len(headers), 1))
    table.style = "Table Grid"
    for index, header in enumerate(headers):
        table.rows[0].cells[index].text = str(header)
        for paragraph in table.rows[0].cells[index].paragraphs:
            for run in paragraph.runs:
                run.bold = True
    for row in rows:
        cells = table.add_row().cells
        for index, value in enumerate(row):
            cells[index].text = value
    doc.add_paragraph()


def render_document(
    document: WordDocument,
    output_path: str | Path,
    metadata: dict[str, str] | None = None,
    footer_text: str | None = None,
    core_properties: dict[str, str] | None = None,
) -> Path:
    """Render a WordDocument to a .docx file."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    doc = Document()

    doc.add_heading(document.title, level=0)
    subtitle = doc.add_paragraph(document.subtitle)
    for run in subtitle.runs:
        run.italic = True
        run.font.size = Pt(12)

    if metadata:
        add_table(doc, ["Field", "Value"], [[key, value] for key, value in metadata.items()])

    doc.add_heading("Summary", level=1)
    doc.add_paragraph(document.summary)

    for index, section in enumerate(document.sections, start=1):
        doc.add_heading(f"{index}. {section.heading}", level=1)
        for paragraph in section.paragraphs:
            doc.add_paragraph(paragraph)
        for bullet in section.bullets:
            doc.add_paragraph(bullet, style="List Bullet")

    doc.add_heading(document.key_data.caption, level=2)
    add_table(doc, document.key_data.columns, normalize_rows(document.key_data))

    if footer_text:
        footer = doc.sections[0].footer.paragraphs[0]
        footer.text = footer_text
        footer.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # Word core properties travel with the file, and plenty of enterprise
    # tooling reads them. Filling them from columns costs nothing.
    for key, value in (core_properties or {}).items():
        if hasattr(doc.core_properties, key) and value is not None:
            setattr(doc.core_properties, key, str(value))

    doc.save(str(output_path))
    return output_path

Structured columns come back as JSON, so we parse them straight into the model that produced them.


In [ ]:
def render_row(row, output_dir: str | Path) -> Path:
    """Render one dataset row to a .docx file, carrying its metadata across."""
    value = row["document"]
    document = WordDocument.model_validate_json(value) if isinstance(value, str) else WordDocument.model_validate(value)
    return render_document(
        document,
        Path(output_dir) / safe_filename(f"{row['doc_id']}-{row['doc_type']}"),
        metadata={
            "Document ID": row["doc_id"],
            "Department": row["department"],
            "Classification": row["classification"],
        },
        footer_text=f"{row['company']} · {row['classification']} · {row['doc_id']}",
        core_properties={"author": row["company"], "category": row["doc_type"]},
    )


preview_path = render_row(preview.dataset.iloc[0], "word-documents-preview")
print(f"📄 {preview_path}  ({preview_path.stat().st_size:,} bytes)")

### 👀 Read it back

Open the file in Word — but also read it back programmatically, because that is how downstream pipelines
will see it. The headings, tables, footer, and core properties are all separately addressable.

Those last two matter more than they look. Putting the classification label in the footer is realistic, and
it is precisely the case where a naive text extractor loses the label entirely. That is a test worth having.


In [ ]:
rendered = Document(str(preview_path))

for paragraph in rendered.paragraphs:
    if paragraph.style.name.startswith(("Title", "Heading")):
        print(f"[{paragraph.style.name}] {paragraph.text}")

print(f"\nkey data     {[cell.text for cell in rendered.tables[-1].rows[0].cells]}")
print(f"footer       {rendered.sections[0].footer.paragraphs[0].text}")
print(f"core author  {rendered.core_properties.author}")

### 🆙 Scale up!

Happy with the preview? Generate the corpus and render all of it.


In [ ]:
results = data_designer.create(config_builder, num_records=3, dataset_name="tutorial-7-word-documents")

dataset = results.load_dataset()
dataset["docx_path"] = [str(render_row(row, "word-documents")) for _, row in dataset.iterrows()]

dataset[["doc_id", "company", "department", "doc_type", "classification", "docx_path"]]

### 📊 The corpus is labelled by construction

Every `.docx` has a row, every row has a `docx_path`, and both carry the sampler controls that produced
them. No annotation pass required — the thin tail you actually wanted to test against is a filter away.


In [ ]:
dataset[dataset["classification"].isin(["Confidential", "Restricted"])][
    ["doc_id", "department", "doc_type", "classification"]
]

## 🔌 Outgrowing the notebook

The loop above is fine for a notebook. It is not fine for 50,000 documents, because it only starts after
the entire dataset has finished generating and it lives outside the Data Designer config.

The production shape is a [processor plugin](https://docs.nvidia.com/nemo/datadesigner/concepts/processors),
and you do not have to write one — this renderer is packaged as
[`data-designer-docx`](https://github.com/NVIDIA-NeMo/DataDesignerPlugins/tree/main/plugins/data-designer-docx)
in the NeMo Data Designer Plugins repository. Once installed, rendering happens *inside* the pipeline:

```python
from data_designer_docx.config import DocxProcessorConfig

config_builder.add_processor(
    DocxProcessorConfig(
        name="word-documents",
        document_column="document",
        filename_template="{{ doc_id }}-{{ doc_type }}.docx",
        metadata_columns={"Document ID": "{{ doc_id }}", "Classification": "{{ classification }}"},
        footer_template="{{ company }} · {{ classification }} · {{ doc_id }}",
    )
)
```

Files then stream out as each batch completes rather than after everything finishes, the path lands back in
the dataset automatically, and the rendering rules travel with the config instead of living in a cell
someone has to remember to run.


## ⏭️ Next Steps

- [Processors](https://docs.nvidia.com/nemo/datadesigner/concepts/processors) — what runs at which stage

- [Plugins overview](https://docs.nvidia.com/nemo/datadesigner/plugins/overview) — writing your own

- The pattern generalizes: swap `python-docx` for `python-pptx` and you generate slide decks; swap it for a
  PDF renderer and, combined with [image columns](https://docs.nvidia.com/nemo/datadesigner/tutorials/generating-images),
  you get the scanned-document corpora that VLM document-understanding work runs on. The structure stays,
  only the renderer changes.
